In [1]:
N = 1500  # for example

special_tokens = ["_", "[PAD]", "[UNK]", "[BOS]", "[EOS]"]
letters = [chr(i) for i in range(ord("a"), ord("z") + 1)]
numbers = [str(i) for i in range(0, N + 1)]
other_tokens = ["|", "?"]

all_tokens = special_tokens + letters + numbers + other_tokens

# Build vocab dict: token -> id
vocab = {tok: i for i, tok in enumerate(all_tokens)}
# vocab['_'] = -100
vocab

{'_': 0,
 '[PAD]': 1,
 '[UNK]': 2,
 '[BOS]': 3,
 '[EOS]': 4,
 'a': 5,
 'b': 6,
 'c': 7,
 'd': 8,
 'e': 9,
 'f': 10,
 'g': 11,
 'h': 12,
 'i': 13,
 'j': 14,
 'k': 15,
 'l': 16,
 'm': 17,
 'n': 18,
 'o': 19,
 'p': 20,
 'q': 21,
 'r': 22,
 's': 23,
 't': 24,
 'u': 25,
 'v': 26,
 'w': 27,
 'x': 28,
 'y': 29,
 'z': 30,
 '0': 31,
 '1': 32,
 '2': 33,
 '3': 34,
 '4': 35,
 '5': 36,
 '6': 37,
 '7': 38,
 '8': 39,
 '9': 40,
 '10': 41,
 '11': 42,
 '12': 43,
 '13': 44,
 '14': 45,
 '15': 46,
 '16': 47,
 '17': 48,
 '18': 49,
 '19': 50,
 '20': 51,
 '21': 52,
 '22': 53,
 '23': 54,
 '24': 55,
 '25': 56,
 '26': 57,
 '27': 58,
 '28': 59,
 '29': 60,
 '30': 61,
 '31': 62,
 '32': 63,
 '33': 64,
 '34': 65,
 '35': 66,
 '36': 67,
 '37': 68,
 '38': 69,
 '39': 70,
 '40': 71,
 '41': 72,
 '42': 73,
 '43': 74,
 '44': 75,
 '45': 76,
 '46': 77,
 '47': 78,
 '48': 79,
 '49': 80,
 '50': 81,
 '51': 82,
 '52': 83,
 '53': 84,
 '54': 85,
 '55': 86,
 '56': 87,
 '57': 88,
 '58': 89,
 '59': 90,
 '60': 91,
 '61': 92,
 '62': 93,
 

In [2]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase, NFD, StripAccents, Sequence as NormSequence

# 1. Create the internal tokenizer
wordlevel = WordLevel(vocab=vocab, unk_token="[UNK]")
tokenizer_backend = Tokenizer(wordlevel)

# 2. Make sure everything is lowercased (optional if your data is already clean)
tokenizer_backend.normalizer = NormSequence([
    NFD(),         # decompose accents
    StripAccents(),
    Lowercase(),
])

# 3. Split by spaces
tokenizer_backend.pre_tokenizer = Whitespace()

from transformers import PreTrainedTokenizerFast

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_backend,
    unk_token="[UNK]",
    pad_token="[PAD]",
    bos_token="[BOS]",
    eos_token="[EOS]",
    mask_token="_",
)

# Sanity check
example = "3 c 15 a 2 b | _ _ _ 15 a 2 b 3"
encoded = hf_tokenizer(example)
print(encoded)
print(hf_tokenizer.convert_ids_to_tokens(encoded["input_ids"]))


/workspace/miniconda3/envs/flame-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'input_ids': [34, 7, 46, 5, 33, 6, 1532, 0, 0, 0, 46, 5, 33, 6, 34], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
['3', 'c', '15', 'a', '2', 'b', '|', '_', '_', '_', '15', 'a', '2', 'b', '3']


In [3]:
hf_tokenizer.mask_token_id

0

In [4]:
import torch
from typing import Any, Dict, List
import numpy as np

def tensorize(example: Dict[str, Any]) -> Dict[str, Any]:
    tensorized = {}
    for key in ['input_ids', 'cu_seqlens']:
        if key not in example:
            continue
        if isinstance(example[key], List):
            tensorized[key] = torch.tensor(example[key], dtype=torch.long)
        elif isinstance(example[key], np.ndarray):
            tensorized[key] = torch.from_numpy(example[key])
        else:
            tensorized[key] = example[key]
    return tensorized

example = tensorize(encoded)
print(example)

{'input_ids': tensor([  34,    7,   46,    5,   33,    6, 1532,    0,    0,    0,   46,    5,
          33,    6,   34])}


In [5]:
example['input_ids'][example['input_ids'] == 0] = -100  # should be -100
print(example)

{'input_ids': tensor([  34,    7,   46,    5,   33,    6, 1532, -100, -100, -100,   46,    5,
          33,    6,   34])}


In [6]:
encoded['input_ids'][encoded['input_ids'] == 0] = -100  # should be -100
print(encoded)

{'input_ids': [-100, 7, 46, 5, 33, 6, 1532, 0, 0, 0, 46, 5, 33, 6, 34], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [7]:
save_dir = "./tasklets_tokenizer"
hf_tokenizer.save_pretrained(save_dir)

# Later:
from transformers import PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast.from_pretrained(save_dir)


In [ ]:
# upload to hf
from huggingface_hub import HfApi
api = HfApi(token="HF_TOKEN")
api.create_repo("zaydzuhri/tasklets_tokenizer", exist_ok=True)
api.upload_folder(
    folder_path=save_dir,
    repo_id="zaydzuhri/tasklets_tokenizer",
    path_in_repo=".",
    commit_message="Add tokenizer files",
    repo_type="model"
)

CommitInfo(commit_url='https://huggingface.co/zaydzuhri/tasklets_tokenizer/commit/014350791eda6113a82c82023241f77cbda2d7f9', commit_message='Add tokenizer files', commit_description='', oid='014350791eda6113a82c82023241f77cbda2d7f9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/zaydzuhri/tasklets_tokenizer', endpoint='https://huggingface.co', repo_type='model', repo_id='zaydzuhri/tasklets_tokenizer'), pr_revision=None, pr_num=None)